# C2 Function 2 Groundwork – C1 Input Contract and Semantic Representation Preparation

This notebook prepares the integration boundary between **Component 1 (aspect/sentiment analysis)** and **Component 2 (review evidence selection)**.

## Current situation
The final C1 output file/schema is not yet available. Therefore, this notebook uses a **clearly labelled synthetic/mock C1 output** only to test:

1. the expected input contract,
2. stable `review_id` joins,
3. one-to-many aspect/sentiment rows,
4. missing/duplicate/orphan record checks,
5. the data structure required before semantic embeddings and minority-evidence qualification.

> **Important:** The mock C1 rows are integration-test data only. They must not be used as research results, evaluation labels, or evidence of C1 model performance.

## Proposed minimum C1 → C2 contract

Required fields from C1:
- `review_id`
- `aspect_category`
- `sentiment`
- `confidence`

C2 will enrich these rows from the shared review dataset using `review_id`:
- `parent_asin`
- `review_date`
- `review_text`

The same stable `review_id` should be preserved across all components.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

print("C1 integration-groundwork environment ready")


C1 integration-groundwork environment ready


## 1. Resolve Component 2 Paths

This cell resolves the repository location robustly so the notebook can be run from the repository root, `component_2`, or `component_2/notebooks`.


In [2]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "component_2").exists():
    COMPONENT_ROOT = CURRENT_DIR / "component_2"
elif CURRENT_DIR.name == "component_2":
    COMPONENT_ROOT = CURRENT_DIR
elif CURRENT_DIR.name == "notebooks" and CURRENT_DIR.parent.name == "component_2":
    COMPONENT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        f"Could not locate component_2 from current directory: {CURRENT_DIR}"
    )

PILOT_PATH = (
    COMPONENT_ROOT
    / "data"
    / "pilot"
    / "smartphone_pilot_cleaned.parquet"
)

INTEGRATION_RESULTS_DIR = (
    COMPONENT_ROOT
    / "results"
    / "integration"
)

INTEGRATION_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MOCK_C1_PATH = (
    INTEGRATION_RESULTS_DIR
    / "c1_mock_aspect_sentiment.csv"
)

JOINED_MOCK_PATH = (
    INTEGRATION_RESULTS_DIR
    / "c1_mock_joined_with_reviews.csv"
)

print("Current directory:", CURRENT_DIR)
print("Component root:", COMPONENT_ROOT)
print("Pilot exists:", PILOT_PATH.exists())
print("Mock C1 output path:", MOCK_C1_PATH)


Current directory: d:\Research\Repository\J26-DS-349\component_2\notebooks
Component root: d:\Research\Repository\J26-DS-349\component_2
Pilot exists: True
Mock C1 output path: d:\Research\Repository\J26-DS-349\component_2\results\integration\c1_mock_aspect_sentiment.csv


## 2. Load the Shared Smartphone Pilot

The shared review dataset is treated as the source of truth for review-level metadata.

The join key is `review_id`.  
Before testing C1 integration, the notebook verifies that every review has one unique stable ID.


In [3]:
reviews_df = pd.read_parquet(PILOT_PATH)

required_review_columns = [
    "review_id",
    "parent_asin",
    "review_date",
    "title",
    "text",
]

missing_review_columns = [
    col
    for col in required_review_columns
    if col not in reviews_df.columns
]

if missing_review_columns:
    raise ValueError(
        f"Missing required review columns: {missing_review_columns}"
    )

assert reviews_df["review_id"].notna().all()
assert reviews_df["review_id"].is_unique

reviews_df["review_text"] = (
    reviews_df["title"]
    .fillna("")
    .astype(str)
    .str.strip()
    + " "
    + reviews_df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
).str.strip()

print("Reviews:", len(reviews_df))
print("Products:", reviews_df["parent_asin"].nunique())
print("Unique review IDs:", reviews_df["review_id"].nunique())
print("Duplicate review IDs:", reviews_df["review_id"].duplicated().sum())
print("Empty review text:", (reviews_df["review_text"] == "").sum())


Reviews: 896
Products: 10
Unique review IDs: 896
Duplicate review IDs: 0
Empty review text: 0


## 3. Define the C1 → C2 Input Contract

The final C1 implementation may contain additional fields, but C2 will initially depend only on this minimum contract.

### Required
- `review_id`: stable identifier matching the shared review dataset
- `aspect_category`: normalized aspect label such as `battery`, `camera`, or `performance`
- `sentiment`: one of `positive`, `neutral`, or `negative`
- `confidence`: C1 confidence score between 0 and 1

### Optional / enrichment fields
`parent_asin`, `review_date`, and review text do not need to be duplicated by C1 if the same `review_id` is preserved. C2 can recover them from the shared review table.


In [4]:
C1_REQUIRED_COLUMNS = [
    "review_id",
    "aspect_category",
    "sentiment",
    "confidence",
]

ALLOWED_SENTIMENTS = {
    "positive",
    "neutral",
    "negative",
}

def validate_c1_contract(c1_df):
    missing_columns = [
        col
        for col in C1_REQUIRED_COLUMNS
        if col not in c1_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"C1 output is missing required columns: {missing_columns}"
        )

    if c1_df[C1_REQUIRED_COLUMNS].isna().any().any():
        raise ValueError(
            "C1 output contains missing values in required fields."
        )

    invalid_sentiments = set(
        c1_df["sentiment"].astype(str).str.lower().unique()
    ) - ALLOWED_SENTIMENTS

    if invalid_sentiments:
        raise ValueError(
            f"Invalid sentiment labels: {sorted(invalid_sentiments)}"
        )

    if not c1_df["confidence"].between(0, 1).all():
        raise ValueError(
            "C1 confidence values must be between 0 and 1."
        )

    print("C1 contract validation passed.")


## 4. Create a Synthetic C1 Output for Integration Testing

Because the real C1 model output is not yet available, the following data is **synthetically generated and clearly marked as mock data**.

Two aspect/sentiment rows are created per review to test the expected one-to-many relationship:

**one review → multiple aspect/sentiment records**

The aspect labels, sentiment labels, and confidence values below are deterministic placeholders only. They do not represent actual predictions.


In [29]:
import hashlib

ASPECTS = [
    "battery",
    "camera",
    "performance",
    "display",
    "price",
    "design",
]

def deterministic_sentiment(
    review_id,
    aspect,
):
    key = f"{review_id}|{aspect}"

    digest = hashlib.sha256(
        key.encode("utf-8")
    ).hexdigest()

    bucket = int(
        digest[:8],
        16,
    ) % 10

    # Synthetic distribution for integration testing only:
    # ~60% positive
    # ~20% neutral
    # ~20% negative
    if bucket < 6:
        return "positive"
    elif bucket < 8:
        return "neutral"
    else:
        return "negative"


def deterministic_confidence(
    review_id,
    aspect,
):
    key = f"{aspect}|{review_id}|confidence"

    digest = hashlib.sha256(
        key.encode("utf-8")
    ).hexdigest()

    bucket = int(
        digest[:8],
        16,
    ) % 36

    return round(
        0.60 + bucket / 100,
        2,
    )


mock_rows = []

for idx, row in reviews_df.reset_index(
    drop=True
).iterrows():

    for offset in [0, 1]:

        aspect = ASPECTS[
            (idx + offset)
            % len(ASPECTS)
        ]

        sentiment = deterministic_sentiment(
            row["review_id"],
            aspect,
        )

        confidence = deterministic_confidence(
            row["review_id"],
            aspect,
        )

        mock_rows.append(
            {
                "review_id": row["review_id"],
                "aspect_category": aspect,
                "sentiment": sentiment,
                "confidence": confidence,
                "is_mock": True,
                "source": "synthetic_contract_test",
            }
        )

c1_mock_df = pd.DataFrame(
    mock_rows
)

print(
    "Mock C1 rows:",
    len(c1_mock_df)
)

print(
    "Unique review IDs:",
    c1_mock_df[
        "review_id"
    ].nunique()
)

print("Rows per review:")
print(
    c1_mock_df
    .groupby("review_id")
    .size()
    .value_counts()
    .sort_index()
)

display(
    c1_mock_df.head(10)
)

Mock C1 rows: 1792
Unique review IDs: 896
Rows per review:
2    896
Name: count, dtype: int64


,review_id,aspect_category,sentiment,confidence,is_mock,source
0,26c3cd248dc3d01d,battery,positive,0.83,True,synthetic_contract_test
1,26c3cd248dc3d01d,camera,neutral,0.76,True,synthetic_contract_test
2,cd97cd121c0fc4b7,camera,neutral,0.90,True,synthetic_contract_test
3,cd97cd121c0fc4b7,performance,negative,0.93,True,synthetic_contract_test
4,293639dc0ec21eae,performance,negative,0.84,True,synthetic_contract_test
5,293639dc0ec21eae,display,positive,0.80,True,synthetic_contract_test
6,5a4857744877f56c,display,neutral,0.64,True,synthetic_contract_test
7,5a4857744877f56c,price,negative,0.78,True,synthetic_contract_test
8,ca2d0d6e3ea776f8,price,positive,0.72,True,synthetic_contract_test
9,ca2d0d6e3ea776f8,design,neutral,0.66,True,synthetic_contract_test


In [30]:
validate_c1_contract(c1_mock_df)

assert c1_mock_df["is_mock"].all()
assert (
    c1_mock_df["source"]
    == "synthetic_contract_test"
).all()

print("Mock-data labels verified.")


C1 contract validation passed.
Mock-data labels verified.


## 5. Validate Review-ID Coverage Before the Join

A real C1 output must not introduce unknown review IDs.

This check reports:

- C1 review IDs that do not exist in the shared review dataset,
- shared review IDs missing from the C1 output,
- overall review-ID coverage.

For the synthetic integration test, coverage is intentionally generated as 100%.


In [31]:
review_ids = set(
    reviews_df["review_id"]
)

c1_ids = set(
    c1_mock_df["review_id"]
)

orphan_c1_ids = c1_ids - review_ids
missing_from_c1 = review_ids - c1_ids

coverage_pct = (
    len(c1_ids & review_ids)
    / len(review_ids)
    * 100
)

print("Orphan C1 review IDs:", len(orphan_c1_ids))
print("Reviews missing from C1:", len(missing_from_c1))
print(f"Review-ID coverage: {coverage_pct:.2f}%")

assert len(orphan_c1_ids) == 0
assert len(missing_from_c1) == 0

print("Review-ID coverage validation passed.")


Orphan C1 review IDs: 0
Reviews missing from C1: 0
Review-ID coverage: 100.00%
Review-ID coverage validation passed.


## 6. Join C1 Aspect/Sentiment Rows with Shared Review Metadata

The C1 aspect/sentiment table is joined back to the shared reviews using `review_id`.

Because one review can contain multiple aspects, the joined table is expected to contain more rows than the original review table.


In [32]:
review_enrichment_df = reviews_df[
    [
        "review_id",
        "parent_asin",
        "review_date",
        "review_text",
    ]
].copy()

joined_df = c1_mock_df.merge(
    review_enrichment_df,
    on="review_id",
    how="left",
    validate="many_to_one",
)

print("Joined rows:", len(joined_df))
print("Joined review IDs:", joined_df["review_id"].nunique())
print(
    "Missing product IDs:",
    joined_df["parent_asin"].isna().sum(),
)
print(
    "Missing review dates:",
    joined_df["review_date"].isna().sum(),
)
print(
    "Missing review text:",
    joined_df["review_text"].isna().sum(),
)

assert len(joined_df) == len(c1_mock_df)
assert joined_df["parent_asin"].notna().all()
assert joined_df["review_text"].notna().all()

print("C1 → C2 join validation passed.")

display(
    joined_df[
        [
            "review_id",
            "parent_asin",
            "aspect_category",
            "sentiment",
            "confidence",
            "is_mock",
        ]
    ].head(10)
)


Joined rows: 1792
Joined review IDs: 896
Missing product IDs: 0
Missing review dates: 0
Missing review text: 0
C1 → C2 join validation passed.


,review_id,parent_asin,aspect_category,sentiment,confidence,is_mock
0,26c3cd248dc3d01d,B0B8PDLMMS,battery,positive,0.83,True
1,26c3cd248dc3d01d,B0B8PDLMMS,camera,neutral,0.76,True
2,cd97cd121c0fc4b7,B09HFW8GRB,camera,neutral,0.90,True
3,cd97cd121c0fc4b7,B09HFW8GRB,performance,negative,0.93,True
4,293639dc0ec21eae,B09JJCXC2M,performance,negative,0.84,True
5,293639dc0ec21eae,B09JJCXC2M,display,positive,0.80,True
6,5a4857744877f56c,B09JJCXC2M,display,neutral,0.64,True
7,5a4857744877f56c,B09JJCXC2M,price,negative,0.78,True
8,ca2d0d6e3ea776f8,B09PPWVQSB,price,positive,0.72,True
9,ca2d0d6e3ea776f8,B09PPWVQSB,design,neutral,0.66,True


## 7. Inspect the Mock Aspect/Sentiment Structure

The following summaries are **integration diagnostics only**.

They are useful for confirming that downstream code can group by:

- product,
- aspect,
- sentiment,
- review.

They must not be interpreted as findings about the Amazon review dataset because the labels are synthetic.


In [33]:
print("Mock aspect counts:")
display(
    joined_df[
        "aspect_category"
    ]
    .value_counts()
    .rename_axis("aspect_category")
    .reset_index(name="mock_count")
)

print("\nMock sentiment counts:")
display(
    joined_df[
        "sentiment"
    ]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="mock_count")
)


Mock aspect counts:


,aspect_category,mock_count
0,camera,300
1,battery,299
2,performance,299
3,display,298
4,price,298
5,design,298



Mock sentiment counts:


,sentiment,mock_count
0,positive,1064
1,negative,367
2,neutral,361


## 8. Save Integration-Test Artifacts

These files make the C1 contract test reproducible.

They remain clearly labelled as mock/synthetic data and should be replaced by the real C1 output when it becomes available.


In [34]:
c1_mock_df.to_csv(
    MOCK_C1_PATH,
    index=False,
)

joined_df.to_csv(
    JOINED_MOCK_PATH,
    index=False,
)

print("Mock C1 output saved:", MOCK_C1_PATH)
print("Joined integration table saved:", JOINED_MOCK_PATH)


Mock C1 output saved: d:\Research\Repository\J26-DS-349\component_2\results\integration\c1_mock_aspect_sentiment.csv
Joined integration table saved: d:\Research\Repository\J26-DS-349\component_2\results\integration\c1_mock_joined_with_reviews.csv


## 9. Contract Replacement Rule for the Real C1 Output

When Component 1 provides its real output:

1. Replace the synthetic/mock C1 table with the real C1 file.
2. Preserve or map the stable `review_id`.
3. Map C1 fields to:
   - `aspect_category`
   - `sentiment`
   - `confidence`
4. Run `validate_c1_contract(...)`.
5. Run the review-ID coverage checks.
6. Join with the shared review dataset.
7. Confirm there are no orphan IDs or missing required metadata.
8. Only then proceed to semantic embeddings and supported-minority analysis.

No mock aspect/sentiment values should be used in research evaluation.


## 10. Current Groundwork Status

At this stage, the following integration behaviour is testable:

- stable review-ID contract,
- C1 required-field validation,
- confidence-range validation,
- sentiment-label validation,
- one-review-to-many-aspect representation,
- review-ID coverage checks,
- orphan-ID detection,
- many-to-one metadata join,
- explicit mock-data labelling.

### Next step
Add **semantic review representations** using a Sentence-BERT model on the real review text.

The semantic embeddings will represent review meaning and will later support:
- semantic representativeness,
- redundancy control,
- similarity-based evidence selection.

The C1 labels used in this notebook are synthetic placeholders only; the review text itself remains real pilot data.


## 11. Generate Semantic Review Embeddings

Sentence-BERT is used to convert each real review into a dense semantic vector.

These embeddings represent the semantic meaning of each review and will later support:
- semantic representativeness,
- similarity computation,
- redundancy reduction,
- evidence selection.

The embeddings are generated from the real review text. Synthetic C1 aspect/sentiment labels are not used to create the embeddings.

In [35]:
from sentence_transformers import SentenceTransformer

In [36]:
SBERT_MODEL_NAME = "all-MiniLM-L6-v2"

sbert_model = SentenceTransformer(
    SBERT_MODEL_NAME
)

print(
    "Loaded Sentence-BERT model:",
    SBERT_MODEL_NAME
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded Sentence-BERT model: all-MiniLM-L6-v2


In [37]:
review_texts = (
    reviews_df["review_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

review_embeddings = sbert_model.encode(
    review_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(
    "Embedding matrix shape:",
    review_embeddings.shape
)

print(
    "Number of reviews:",
    len(reviews_df)
)

print(
    "Embedding dimension:",
    review_embeddings.shape[1]
)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Embedding matrix shape: (896, 384)
Number of reviews: 896
Embedding dimension: 384


In [38]:
assert (
    review_embeddings.shape[0]
    == len(reviews_df)
)

assert (
    review_embeddings.shape[1]
    == 384
)

assert np.isfinite(
    review_embeddings
).all()

print(
    "Semantic embedding validation passed."
)

Semantic embedding validation passed.


In [39]:
EMBEDDING_PATH = (
    INTEGRATION_RESULTS_DIR
    / "smartphone_review_embeddings.npy"
)

EMBEDDING_INDEX_PATH = (
    INTEGRATION_RESULTS_DIR
    / "smartphone_review_embedding_index.csv"
)

np.save(
    EMBEDDING_PATH,
    review_embeddings,
)

embedding_index_df = reviews_df[
    [
        "review_id",
        "parent_asin",
        "review_date",
    ]
].copy()

embedding_index_df[
    "embedding_row"
] = np.arange(
    len(embedding_index_df)
)

embedding_index_df.to_csv(
    EMBEDDING_INDEX_PATH,
    index=False,
)

print(
    "Embeddings saved:",
    EMBEDDING_PATH
)

print(
    "Embedding index saved:",
    EMBEDDING_INDEX_PATH
)

Embeddings saved: d:\Research\Repository\J26-DS-349\component_2\results\integration\smartphone_review_embeddings.npy
Embedding index saved: d:\Research\Repository\J26-DS-349\component_2\results\integration\smartphone_review_embedding_index.csv


In [40]:
loaded_embeddings = np.load(
    EMBEDDING_PATH
)

loaded_index_df = pd.read_csv(
    EMBEDDING_INDEX_PATH
)

print(
    "Loaded embeddings shape:",
    loaded_embeddings.shape
)

print(
    "Loaded index rows:",
    len(loaded_index_df)
)

assert (
    loaded_embeddings.shape[0]
    == len(loaded_index_df)
)

assert (
    loaded_index_df["review_id"].nunique()
    == len(reviews_df)
)

print(
    "Saved embedding artifacts validated."
)

Loaded embeddings shape: (896, 384)
Loaded index rows: 896
Saved embedding artifacts validated.


## 12. Semantic Similarity and Redundancy Diagnostics

Cosine similarity is used to compare semantic review embeddings.

This step provides preliminary evidence for the redundancy-control part of the review selector.

Higher cosine similarity indicates that two reviews express more similar semantic content.

The purpose of this diagnostic is to:
- identify highly similar review pairs,
- inspect potential redundant evidence,
- prepare similarity-based penalties for the constrained selector.

No final redundancy threshold is fixed at this stage. Thresholds will be examined empirically and later validated through sensitivity analysis.

In [41]:
from sklearn.metrics.pairwise import cosine_similarity

In [42]:
product_similarity_records = []

for product_id, product_group in reviews_df.groupby(
    "parent_asin"
):
    product_indices = product_group.index.to_numpy()

    product_embeddings = review_embeddings[
        product_indices
    ]

    similarity_matrix = cosine_similarity(
        product_embeddings
    )

    upper_triangle = np.triu_indices(
        similarity_matrix.shape[0],
        k=1,
    )

    pairwise_scores = similarity_matrix[
        upper_triangle
    ]

    if len(pairwise_scores) == 0:
        continue

    product_similarity_records.append(
        {
            "parent_asin": product_id,
            "num_reviews": len(product_group),
            "mean_similarity": float(
                np.mean(pairwise_scores)
            ),
            "median_similarity": float(
                np.median(pairwise_scores)
            ),
            "max_similarity": float(
                np.max(pairwise_scores)
            ),
        }
    )

product_similarity_df = pd.DataFrame(
    product_similarity_records
)

display(product_similarity_df)

,parent_asin,num_reviews,mean_similarity,median_similarity,max_similarity
0,B07GYY6VTD,100,0.381112,0.392600,0.967796
1,B07SDQ483V,83,0.369231,0.371652,0.875108
2,B08PP9FM9N,61,0.361806,0.372089,0.779178
3,B08XRDVWQ2,100,0.361499,0.370881,0.845886
4,B09HFW8GRB,77,0.383560,0.381926,0.814245
5,B09JJCXC2M,100,0.353726,0.354883,0.868574
6,B09PPWVQSB,100,0.355999,0.360254,0.827413
7,B0B8PDLMMS,75,0.348900,0.344881,0.795621
8,B0BL9TKSBL,100,0.412338,0.415822,0.894732
9,B0BMWTL2S6,100,0.351596,0.355389,0.825028


In [43]:
high_similarity_pairs = []

SIMILARITY_INSPECTION_THRESHOLD = 0.85

for product_id, product_group in reviews_df.groupby(
    "parent_asin"
):
    product_indices = product_group.index.to_numpy()

    product_embeddings = review_embeddings[
        product_indices
    ]

    similarity_matrix = cosine_similarity(
        product_embeddings
    )

    local_indices = list(
        range(len(product_group))
    )

    product_group_reset = product_group.reset_index(
        drop=True
    )

    for i in local_indices:
        for j in range(i + 1, len(local_indices)):
            similarity = similarity_matrix[i, j]

            if (
                similarity
                >= SIMILARITY_INSPECTION_THRESHOLD
            ):
                high_similarity_pairs.append(
                    {
                        "parent_asin": product_id,
                        "review_id_1": product_group_reset.loc[
                            i,
                            "review_id",
                        ],
                        "review_id_2": product_group_reset.loc[
                            j,
                            "review_id",
                        ],
                        "cosine_similarity": float(
                            similarity
                        ),
                        "text_1": product_group_reset.loc[
                            i,
                            "review_text",
                        ],
                        "text_2": product_group_reset.loc[
                            j,
                            "review_text",
                        ],
                    }
                )

high_similarity_df = pd.DataFrame(
    high_similarity_pairs
)

print(
    "Highly similar review pairs:",
    len(high_similarity_df)
)

display(
    high_similarity_df.sort_values(
        "cosine_similarity",
        ascending=False,
    ).head(10)
)

Highly similar review pairs: 6


,parent_asin,review_id_1,review_id_2,cosine_similarity,text_1,text_2
0,B07GYY6VTD,5fd5431e6b99f4b6,041647622c7b6b96,0.967796,Five Stars works with tracfone,Five Stars worked with tracfone
5,B0BL9TKSBL,d3cd5eaca52a6735,aac4fcebed2025a5,0.894732,I love this phone. Best ever. Great phone,Love everything about this phone. Worth every ...
2,B07SDQ483V,fac30ea8466d41d5,20d8e9690a7ba21e,0.875108,Meh Not a good phone. 2nd Blu purchase 1st was...,Good phone for a great price. I buy Blu phones...
3,B07SDQ483V,03fe2e44ba4656ed,20d8e9690a7ba21e,0.873844,Great phone at a great price. This is my secon...,Good phone for a great price. I buy Blu phones...
4,B09JJCXC2M,5cbebac5455ccdc4,b5af647105392147,0.868574,Great phone and great condition Perfect condit...,Smartphone Excellent condition and good price....
1,B07SDQ483V,fac30ea8466d41d5,03fe2e44ba4656ed,0.855602,Meh Not a good phone. 2nd Blu purchase 1st was...,Great phone at a great price. This is my secon...


In [44]:
SIMILARITY_SUMMARY_PATH = (
    INTEGRATION_RESULTS_DIR
    / "semantic_similarity_summary.csv"
)

HIGH_SIMILARITY_PATH = (
    INTEGRATION_RESULTS_DIR
    / "high_similarity_review_pairs.csv"
)

product_similarity_df.to_csv(
    SIMILARITY_SUMMARY_PATH,
    index=False,
)

high_similarity_df.to_csv(
    HIGH_SIMILARITY_PATH,
    index=False,
)

print(
    "Similarity summary saved:",
    SIMILARITY_SUMMARY_PATH
)

print(
    "High-similarity pairs saved:",
    HIGH_SIMILARITY_PATH
)

Similarity summary saved: d:\Research\Repository\J26-DS-349\component_2\results\integration\semantic_similarity_summary.csv
High-similarity pairs saved: d:\Research\Repository\J26-DS-349\component_2\results\integration\high_similarity_review_pairs.csv


## 13. Prepare a Reusable Redundancy Penalty

The evidence selector should avoid repeatedly selecting reviews that express almost the same semantic information.

For a candidate review, redundancy is defined as its maximum cosine similarity to any review already selected.

- If no review has been selected yet, redundancy is 0.
- Higher similarity means stronger redundancy.
- The final constrained selector will subtract a weighted redundancy penalty from the candidate's selection score.

The inspection threshold of 0.85 used earlier is not required by this penalty and is not treated as a final research threshold.

In [45]:
embedding_lookup_df = embedding_index_df[
    [
        "review_id",
        "embedding_row",
    ]
].copy()

reviews_with_embedding_df = reviews_df.merge(
    embedding_lookup_df,
    on="review_id",
    how="left",
    validate="one_to_one",
)

assert (
    reviews_with_embedding_df[
        "embedding_row"
    ].isna().sum()
    == 0
)

reviews_with_embedding_df[
    "embedding_row"
] = (
    reviews_with_embedding_df[
        "embedding_row"
    ].astype(int)
)

print(
    "Reviews mapped to embeddings:",
    len(reviews_with_embedding_df)
)

print(
    "Unique embedding rows:",
    reviews_with_embedding_df[
        "embedding_row"
    ].nunique()
)

print(
    "Embedding-row mapping validation passed."
)

Reviews mapped to embeddings: 896
Unique embedding rows: 896
Embedding-row mapping validation passed.


In [46]:
def candidate_redundancy(
    candidate_embedding_row,
    selected_embedding_rows,
    embeddings,
):
    if len(selected_embedding_rows) == 0:
        return 0.0

    candidate_vector = embeddings[
        candidate_embedding_row
    ].reshape(1, -1)

    selected_vectors = embeddings[
        selected_embedding_rows
    ]

    similarities = cosine_similarity(
        candidate_vector,
        selected_vectors,
    )[0]

    return float(
        np.max(similarities)
    )

In [47]:
sample_product = (
    reviews_with_embedding_df[
        reviews_with_embedding_df[
            "parent_asin"
        ] == "B07GYY6VTD"
    ]
    .reset_index(drop=True)
)

candidate_row = int(
    sample_product.loc[
        1,
        "embedding_row",
    ]
)

selected_rows = [
    int(
        sample_product.loc[
            0,
            "embedding_row",
        ]
    )
]

sample_redundancy = candidate_redundancy(
    candidate_row,
    selected_rows,
    review_embeddings,
)

print(
    "Sample candidate redundancy:",
    round(
        sample_redundancy,
        4,
    ),
)

assert 0.0 <= sample_redundancy <= 1.0001

print(
    "Redundancy penalty function validation passed."
)

Sample candidate redundancy: 0.4393
Redundancy penalty function validation passed.


## 14. Supported Minority Aspect-Sentiment Qualification

Component 2 should preserve minority opinions only when they have sufficient evidence.

A minority aspect-sentiment group is not automatically treated as important simply because it is rare. It must first satisfy support requirements.

This step calculates, within each product and aspect:

- number of reviews supporting each sentiment,
- sentiment prevalence,
- average C1 confidence,
- whether the group is sufficiently supported,
- whether the group is a minority candidate.

Important:
The current C1 aspect/sentiment labels are synthetic integration-test data. Therefore, the results produced in this section are used only to validate the qualification pipeline and must not be interpreted as research findings.

Final thresholds will be determined later using real C1 outputs, EDA, and sensitivity analysis.

In [48]:
MIN_SUPPORT_COUNT = 3
MIN_AVG_CONFIDENCE = 0.60
MINORITY_MAX_SHARE = 0.40

print("Integration-test thresholds")
print("Minimum support count:", MIN_SUPPORT_COUNT)
print("Minimum average confidence:", MIN_AVG_CONFIDENCE)
print("Minority maximum prevalence:", MINORITY_MAX_SHARE)

print(
    "\nThese are provisional integration-test values, "
    "not final research thresholds."
)

Integration-test thresholds
Minimum support count: 3
Minimum average confidence: 0.6
Minority maximum prevalence: 0.4

These are provisional integration-test values, not final research thresholds.


In [49]:
group_support_df = (
    joined_df
    .groupby(
        [
            "parent_asin",
            "aspect_category",
            "sentiment",
        ],
        as_index=False,
    )
    .agg(
        support_count=(
            "review_id",
            "nunique",
        ),
        avg_confidence=(
            "confidence",
            "mean",
        ),
    )
)

aspect_totals_df = (
    group_support_df
    .groupby(
        [
            "parent_asin",
            "aspect_category",
        ],
        as_index=False,
    )
    .agg(
        aspect_total_support=(
            "support_count",
            "sum",
        )
    )
)

minority_stats_df = group_support_df.merge(
    aspect_totals_df,
    on=[
        "parent_asin",
        "aspect_category",
    ],
    how="left",
    validate="many_to_one",
)

minority_stats_df[
    "prevalence"
] = (
    minority_stats_df[
        "support_count"
    ]
    / minority_stats_df[
        "aspect_total_support"
    ]
)

display(
    minority_stats_df.head(15)
)

,parent_asin,aspect_category,sentiment,support_count,avg_confidence,aspect_total_support,prevalence
0,B07GYY6VTD,battery,negative,8,0.791250,30,0.266667
1,B07GYY6VTD,battery,neutral,8,0.770000,30,0.266667
2,B07GYY6VTD,battery,positive,14,0.803571,30,0.466667
3,B07GYY6VTD,camera,negative,3,0.673333,30,0.100000
4,B07GYY6VTD,camera,neutral,9,0.862222,30,0.300000
5,B07GYY6VTD,camera,positive,18,0.813889,30,0.600000
6,B07GYY6VTD,design,negative,9,0.798889,33,0.272727
7,B07GYY6VTD,design,neutral,2,0.940000,33,0.060606
8,B07GYY6VTD,design,positive,22,0.806364,33,0.666667
9,B07GYY6VTD,display,negative,4,0.757500,37,0.108108


In [50]:
minority_stats_df[
    "sufficient_support"
] = (
    (
        minority_stats_df[
            "support_count"
        ]
        >= MIN_SUPPORT_COUNT
    )
    &
    (
        minority_stats_df[
            "avg_confidence"
        ]
        >= MIN_AVG_CONFIDENCE
    )
)

minority_stats_df[
    "minority_candidate"
] = (
    minority_stats_df[
        "prevalence"
    ]
    <= MINORITY_MAX_SHARE
)

minority_stats_df[
    "supported_minority"
] = (
    minority_stats_df[
        "sufficient_support"
    ]
    &
    minority_stats_df[
        "minority_candidate"
    ]
)

print(
    "Total aspect-sentiment groups:",
    len(minority_stats_df)
)

print(
    "Sufficiently supported groups:",
    minority_stats_df[
        "sufficient_support"
    ].sum()
)

print(
    "Minority candidates:",
    minority_stats_df[
        "minority_candidate"
    ].sum()
)

print(
    "Supported minority groups:",
    minority_stats_df[
        "supported_minority"
    ].sum()
)

Total aspect-sentiment groups: 180
Sufficiently supported groups: 170
Minority candidates: 121
Supported minority groups: 111


In [51]:
supported_minority_df = (
    minority_stats_df[
        minority_stats_df[
            "supported_minority"
        ]
    ]
    .sort_values(
        [
            "parent_asin",
            "aspect_category",
            "prevalence",
        ]
    )
    .reset_index(drop=True)
)

display(
    supported_minority_df.head(20)
)

,parent_asin,aspect_category,sentiment,support_count,avg_confidence,aspect_total_support,prevalence,sufficient_support,minority_candidate,supported_minority
0,B07GYY6VTD,battery,negative,8,0.791250,30,0.266667,True,True,True
1,B07GYY6VTD,battery,neutral,8,0.770000,30,0.266667,True,True,True
2,B07GYY6VTD,camera,negative,3,0.673333,30,0.100000,True,True,True
3,B07GYY6VTD,camera,neutral,9,0.862222,30,0.300000,True,True,True
4,B07GYY6VTD,design,negative,9,0.798889,33,0.272727,True,True,True
5,B07GYY6VTD,display,negative,4,0.757500,37,0.108108,True,True,True
6,B07GYY6VTD,display,neutral,6,0.730000,37,0.162162,True,True,True
7,B07GYY6VTD,performance,neutral,4,0.805000,39,0.102564,True,True,True
8,B07GYY6VTD,performance,negative,9,0.782222,39,0.230769,True,True,True
9,B07GYY6VTD,price,neutral,5,0.786000,31,0.161290,True,True,True


In [52]:
assert (
    minority_stats_df[
        "prevalence"
    ].between(0, 1).all()
)

assert (
    minority_stats_df[
        "avg_confidence"
    ].between(0, 1).all()
)

assert (
    minority_stats_df[
        "supported_minority"
    ]
    <= minority_stats_df[
        "sufficient_support"
    ]
).all()

print(
    "Supported-minority qualification "
    "pipeline validation passed."
)

Supported-minority qualification pipeline validation passed.


In [53]:
MINORITY_STATS_PATH = (
    INTEGRATION_RESULTS_DIR
    / "mock_minority_qualification_stats.csv"
)

SUPPORTED_MINORITY_PATH = (
    INTEGRATION_RESULTS_DIR
    / "mock_supported_minority_groups.csv"
)

minority_stats_df.to_csv(
    MINORITY_STATS_PATH,
    index=False,
)

supported_minority_df.to_csv(
    SUPPORTED_MINORITY_PATH,
    index=False,
)

print(
    "Minority qualification stats saved:",
    MINORITY_STATS_PATH,
)

print(
    "Supported minority groups saved:",
    SUPPORTED_MINORITY_PATH,
)

Minority qualification stats saved: d:\Research\Repository\J26-DS-349\component_2\results\integration\mock_minority_qualification_stats.csv
Supported minority groups saved: d:\Research\Repository\J26-DS-349\component_2\results\integration\mock_supported_minority_groups.csv


## 15. Function 2 Groundwork Summary

Completed in this notebook:

- Defined the provisional C1 → C2 input contract.
- Validated stable `review_id` integration.
- Tested one-review-to-many-aspect structure using clearly labelled synthetic C1 data.
- Validated C1 coverage and metadata joins.
- Generated Sentence-BERT embeddings for all 896 real smartphone pilot reviews.
- Saved and reloaded the 384-dimensional semantic embeddings.
- Calculated within-product semantic similarity.
- Identified high-similarity review pairs for redundancy diagnostics.
- Implemented a reusable semantic redundancy penalty.
- Implemented and validated the supported-minority qualification pipeline.

### Important limitation

The aspect/sentiment labels used for integration and minority-qualification testing are synthetic placeholders because the final C1 model output is not yet available.

Therefore, mock aspect counts, sentiment counts, minority counts, and supported-minority counts are not research results.

### Next step

Proceed to the constrained fixed-budget review selector, which will combine:

- predicted usefulness,
- semantic representativeness,
- aspect coverage,
- redundancy control,
- supported-minority preservation,
- fixed review budget.

The selector will be implemented separately in `05_supported_minority_selector.ipynb`.